# Twitter language-separation embeddings in 2D

Load the prepared EmbeddingGemma 2 vectors, optionally tune IMSTE to separate language classes, review and override the suggested settings, then compare UMAP and language-supervised IMSTE.

Run notebooks/dataset_setup/prepare_twitter_dataset.ipynb first. Balanced sampling by language is recommended so the layout and tuning see comparable class counts. The plot can still be colored by source batch, language, posting time, retweet status or count, hashtag, or engagement.

## 1. Load the prepared vectors

In [11]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
from IPython.display import display
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file() and (p / "imste").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the IMSTE repository.")
for path in (PROJECT_ROOT, PROJECT_ROOT / "notebooks" / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import imste
if not Path(imste.__file__).resolve().is_relative_to(PROJECT_ROOT.resolve()):
    raise RuntimeError("Restart the kernel to load IMSTE from this checkout.")
from twitter_dataset import load_twitter_dataset
DATA_DIR = PROJECT_ROOT / "data" / "twitter"
tweets_all, X_all, data_manifest = load_twitter_dataset(DATA_DIR)
print(f"Loaded {len(tweets_all):,} tweets × {X_all.shape[1]:,} features")
print("Embedding model:", data_manifest["embedding_model"])
print("Dataset ID:", data_manifest["dataset_id"])

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loaded 1,988,195 tweets × 384 features
Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Dataset ID: 7dc37cf51c52345bc93f


## 2. Choose layout sample and point colors

Set a cap for a quick first look. The selected subset is deterministic.

In [12]:
RUN_IMSTE_TUNING = False
MAX_TWEETS_FOR_LAYOUT = None  # Set None to use all cached tweets
COLOR_BY = "language"      # source_batch, language, hour, weekday, retweet, retweet_count, hashtag, engagement
TOP_HASHTAGS = 12

In [13]:
RANDOM_STATE = int(data_manifest["random_state"])
tweets = tweets_all.copy()
X = X_all
if MAX_TWEETS_FOR_LAYOUT is not None:
    if not isinstance(MAX_TWEETS_FOR_LAYOUT, int) or MAX_TWEETS_FOR_LAYOUT < 2:
        raise ValueError("MAX_TWEETS_FOR_LAYOUT must be at least 2 or None.")
    if len(tweets) > MAX_TWEETS_FOR_LAYOUT:
        selected = np.sort(np.random.default_rng(RANDOM_STATE).choice(
            len(tweets), size=MAX_TWEETS_FOR_LAYOUT, replace=False))
        tweets = tweets.iloc[selected].reset_index(drop=True)
        X = X[selected]

language_target = tweets["lang"].fillna("unknown").astype(str).to_numpy()
language_class_counts = pd.Series(language_target).value_counts()
LANGUAGE_HAS_VARIATION = len(language_class_counts) > 1
TUNABLE_LANGUAGE_CLASSES = language_class_counts[language_class_counts >= 5].index.to_numpy()
if not LANGUAGE_HAS_VARIATION:
    print("Only one language class is present; IMSTE will fit without label supervision.")
elif len(TUNABLE_LANGUAGE_CLASSES) < len(language_class_counts):
    print("Some language classes have fewer than five tweets and will be left out of tuning.")

timestamps = pd.to_datetime(tweets["created_at"], format="mixed", errors="coerce", utc=True)
retweet_counts = pd.to_numeric(tweets["retweet_count"], errors="coerce").fillna(0).clip(lower=0)
if COLOR_BY == "source_batch":
    color_values = tweets["source_batch"].fillna("unknown").astype(str)
elif COLOR_BY == "language":
    color_values = pd.Series(language_target, index=tweets.index)
elif COLOR_BY == "hour":
    color_values = timestamps.dt.hour.map(lambda x: f"{int(x):02d}:00" if pd.notna(x) else "unknown")
elif COLOR_BY == "weekday":
    color_values = timestamps.dt.day_name().fillna("unknown")
elif COLOR_BY == "retweet":
    color_values = tweets["is_retweet"].map({True: "retweet", False: "original"}).fillna("unknown")
elif COLOR_BY == "retweet_count":
    color_values = pd.cut(
        retweet_counts,
        bins=[-0.5, 0.5, 1.5, 5.5, 20.5, 100.5, np.inf],
        labels=["0", "1", "2–5", "6–20", "21–100", ">100"],
    ).astype(str)
elif COLOR_BY == "hashtag":
    hashtags = tweets["top_hashtag"].fillna("(no hashtag)").astype(str)
    no_hashtag = hashtags.eq("(no hashtag)")
    hashtag_counts = hashtags[~no_hashtag].value_counts()
    top = set(hashtag_counts.head(TOP_HASHTAGS).index)
    color_values = hashtags.where(
        no_hashtag | hashtags.isin(top), "Other hashtags"
    ).replace("(no hashtag)", "No hashtag")
elif COLOR_BY == "engagement":
    favorites = pd.to_numeric(tweets["favorite_count"], errors="coerce").fillna(0).clip(lower=0)
    engagement = retweet_counts + favorites
    if engagement.nunique() < 2:
        color_values = pd.Series(f"all tweets: {engagement.iloc[0]:g} engagement", index=tweets.index, dtype="string")
    else:
        color_values = pd.qcut(np.log1p(engagement), q=5, duplicates="drop").astype(str)
else:
    raise ValueError("Unknown COLOR_BY choice.")
counts = color_values.astype(str).value_counts()
if COLOR_BY in {"source_batch", "language", "hashtag"} and len(counts) > 20:
    keep = set(counts.head(19).index)
    color_values = color_values.astype(str).where(color_values.astype(str).isin(keep), "other")
labels = pd.Categorical(color_values.astype(str))
print(f"Using {len(tweets):,} tweets; target=language ({len(language_class_counts)} classes); coloring by {COLOR_BY}.")
display(language_class_counts.rename("tweets").to_frame())


Some language classes have fewer than five tweets and will be left out of tuning.
Using 1,988,195 tweets; target=language (66 classes); coloring by language.


,tweets
ar,80978
en,80976
es,80976
fr,80975
in,80973
...,...
chr,17
ug,12
sd,10
am,7


## 3. Configure UMAP and tune IMSTE

In [14]:
import umap
from imste import IteratedMinimumSpanningTreeEmbedder
METRIC = "euclidean"
UMAP_SETTINGS = {
    "n_components": 2, "n_neighbors": 15, "min_dist": 0.1,
    "metric": METRIC, "random_state": RANDOM_STATE,
    "init": "recursive", "recursive_coarsening_ratio": 4,
    "compatibility_layout": False, "nearest_neighbor_backend": "rinnd",
    "n_epochs": 200, "verbose": True,
}
IMSTE_BASE_PARAMS = {
    "n_msts": 3, "learning_rate": 0.05, "negative_ratio": 5,
    "n_epochs": 200, "label_strength": 0.6,
    "gravity_strength": 0.2, "mean_shift_iterations": 3,
}

TUNE_MAX_PER_CLASS = 30
TUNE_EPOCHS = 60
TUNE_TRIALS = 20
TUNE_SEARCH_SPACE = {
    "n_msts": (2, 32), "negative_ratio": (1, 10),
    "logistic_margin": (0.25, 2.0), "logistic_temperature": (0.1, 1.0),
    "lambda_rep": (0.25, 0.75), "gravity_strength": (0.0, 1.0),
    "gravity_resolution": (0.5, 2.0), "gravity_min_dist": (0.01, 1.0),
    "mean_shift_iterations": (0, 5), "include_knn_attraction": [False, True],
    "label_strength": (0.0, 2.0),
}


In [15]:
tuning_seconds = 0.0
IMSTE_TUNED_PARAMS = {}
if RUN_IMSTE_TUNING and len(TUNABLE_LANGUAGE_CLASSES) >= 2:
    rng = np.random.default_rng(RANDOM_STATE)
    tune_indices = []
    for language in TUNABLE_LANGUAGE_CLASSES:
        class_indices = np.flatnonzero(language_target == language)
        n = min(len(class_indices), TUNE_MAX_PER_CLASS)
        tune_indices.append(rng.choice(class_indices, size=n, replace=False))
    tune_indices = np.sort(np.concatenate(tune_indices))
    X_tune, language_tune = X[tune_indices], language_target[tune_indices]
    print(f"Tuning on {len(tune_indices):,} tweets across {len(TUNABLE_LANGUAGE_CLASSES)} language classes")
    tune_mapper = IteratedMinimumSpanningTreeEmbedder(
        **{**IMSTE_BASE_PARAMS, "n_epochs": TUNE_EPOCHS,
           "random_state": RANDOM_STATE, "metric": METRIC, "verbose": True}
    )
    started = time.perf_counter()
    tune_result = tune_mapper.tune(
        X_tune, language_tune, task="classification", scoring="balanced_accuracy",
        search_space=TUNE_SEARCH_SPACE, n_samples=TUNE_TRIALS,
        n_initial_points=min(10, TUNE_TRIALS), coverage_weight=0.4,
    )
    tuning_seconds = time.perf_counter() - started
    IMSTE_TUNED_PARAMS = dict(tune_result.best_params)
    print(
        f"Best CV balanced accuracy: {tune_result.best_score:.3f}; "
        f"accuracy: {tune_result.best_accuracy:.3f}; "
        f"coverage: {tune_result.best_coverage:.3f}"
    )
    trials = pd.DataFrame(tune_result.x_iters, columns=TUNE_SEARCH_SPACE)
    trials["balanced_accuracy"] = tune_result.cv_scores
    trials["accuracy"] = tune_result.accuracy_scores
    trials["coverage"] = tune_result.coverage_scores
    trials["objective"] = tune_result.objective_scores
    display(trials.sort_values("objective", ascending=False).reset_index(drop=True))
elif RUN_IMSTE_TUNING:
    print("Tuning skipped: at least two language classes need five or more tweets each.")
else:
    print("Tuning skipped by configuration; using the configured language-supervised IMSTE parameters.")


Tuning skipped by configuration; using the configured language-supervised IMSTE parameters.


## 4. Review or override the suggested parameters

Set any values in `IMSTE_OVERRIDES`. These take precedence over the tuner’s suggestion before the full layout fit. `label_strength` controls how strongly same-language graph edges attract relative to different-language edges.

In [16]:
umap_started = time.perf_counter()
Z_umap = umap.UMAP(**UMAP_SETTINGS).fit_transform(X)
umap_seconds = time.perf_counter() - umap_started
print(f"UMAP fit: {umap_seconds / 60:.1f} min")

UMAP(compatibility_layout=False, init='recursive', n_epochs=200, nearest_neighbor_backend='rinnd', random_state=42, verbose=True)
Tue Nov 24 07:44:55 2026 Construct fuzzy simplicial set
Tue Nov 24 07:44:55 2026 Finding Nearest Neighbors
Tue Nov 24 07:44:55 2026 Using RINND to find nearest neighbors
Building RP forest with 8 trees...
Initializing graph from 451488 leaves...
Running NN-descent for up to 21 iterations...
Iteration 1: 18809730 updates
Iteration 2: 18509013 updates
Iteration 3: 9239753 updates
Iteration 4: 4371330 updates
Iteration 5: 2060998 updates
Iteration 6: 984446 updates
Iteration 7: 476705 updates
Iteration 8: 233995 updates
Iteration 9: 117884 updates
Iteration 10: 60545 updates
Iteration 11: 31637 updates
Iteration 12: 16609 updates
Converged after 12 iterations

=== Timing Breakdown ===
Forest building:    4111.343ms
Leaf initialization:3236.721ms
Candidate building: 1192.435ms (12 iters)
Update iterations:  14665.918ms (12 iters)
Total measured:     23206.418ms


/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users

Tue Nov 24 07:45:36 2026 Recursive initialization: coarsening 497048 nodes into about 124262 parts
Tue Nov 24 07:45:39 2026 Recursive initialization: coarsening 124262 nodes into about 31065 parts
Tue Nov 24 07:45:41 2026 Recursive initialization: coarsening 31065 nodes into about 7766 parts
Tue Nov 24 07:45:41 2026 Recursive initialization: coarsening 7766 nodes into about 1941 parts
Tue Nov 24 07:45:41 2026 Recursive initialization: coarsening 1941 nodes into about 485 parts


Init recursion depth 6:   0%|          | 0/275 [00:00<?, ?it/s]

Init recursion depth 5:   0%|          | 0/195 [00:00<?, ?it/s]

/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/umap/label_prop.py:959: RuntimeWarning: divide by zero encountered in matmul
  result = result @ rotation
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/umap/label_prop.py:959: RuntimeWarning: overflow encountered in matmul
  result = result @ rotation
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/umap/label_prop.py:959: RuntimeWarning: invalid value encountered in matmul
  result = result @ rotation


... (3 outputs left out)

Init recursion depth 1:   0%|          | 0/50 [00:00<?, ?it/s]

Tue Nov 24 07:45:47 2026 Using adam layout optimizer


Epochs completed:   0%|            0/200 [00:00]

Tue Nov 24 07:47:23 2026 Finished embedding
UMAP fit: 2.5 min


In [29]:
IMSTE_OVERRIDES = {
    # Examples: 
    "n_msts": 32, 
    "negative_ratio": 1,
    "label_strength": 4.0, 
    "gravity_strength": 0, 
    "mean_shift_iterations": 0,
}
FINAL_IMSTE_PARAMS = {
    **IMSTE_BASE_PARAMS, **IMSTE_TUNED_PARAMS, **IMSTE_OVERRIDES,
    "random_state": RANDOM_STATE, "metric": METRIC, "verbose": True,
}
if not LANGUAGE_HAS_VARIATION:
    FINAL_IMSTE_PARAMS["label_strength"] = 0.0
print("Classification target: language")
print("Tuner best parameters:", IMSTE_TUNED_PARAMS or "(none)")
print("Final parameters after user overrides:", FINAL_IMSTE_PARAMS)


Classification target: language
Tuner best parameters: (none)
Final parameters after user overrides: {'n_msts': 32, 'learning_rate': 0.05, 'negative_ratio': 1, 'n_epochs': 200, 'label_strength': 4.0, 'gravity_strength': 0, 'mean_shift_iterations': 0, 'random_state': 42, 'metric': 'euclidean', 'verbose': True}


In [ ]:

imste_started = time.perf_counter()
imste_estimator = IteratedMinimumSpanningTreeEmbedder(**FINAL_IMSTE_PARAMS)
if LANGUAGE_HAS_VARIATION:
    Z_imste = imste_estimator.fit_transform(X, y=language_target)
    fit_description = "Language-supervised IMSTE fit"
else:
    Z_imste = imste_estimator.fit_transform(X)
    fit_description = "Unsupervised IMSTE fit (only one language class)"
imste_seconds = time.perf_counter() - imste_started
print(f"{fit_description}: {imste_seconds / 60:.1f} min")


Tue Nov 24 11:26:27 2026 IteratedMinimumSpanningTreeEmbedder({'batch_size': 131072, 'device': 'auto', 'epsilon': 0.0001, 'gravity_community_sample_size': 50000, 'gravity_core_quantile': 0.95, 'gravity_mass_power': 0.5, 'gravity_min_dist': 0.0, 'gravity_min_dist_iterations': 32, 'gravity_neighbors': 15, 'gravity_pack_gap_fraction': 0.25, 'gravity_pack_radius_quantile': 0.9, 'gravity_pack_regions': False, 'gravity_resolution': 1.0, 'gravity_strength': 0, 'include_knn_attraction': False, 'input_preprocessor': None, 'label_strength': 4.0, 'lambda_rep': 0.5, 'learning_rate': 0.05, 'logistic_margin': 1.0, 'logistic_temperature': 0.5, 'mean_shift_bandwidth_factor': 5.0, 'mean_shift_iterations': 0, 'mean_shift_neighbors': 100, 'mean_shift_step_size': 0.5, 'metric': 'euclidean', 'mst_inter_component_edges': 5, 'mst_max_neighbors': None, 'mst_method': 'famst', 'mst_neighbor_backend': 'auto', 'mst_neighbors': 15, 'mst_representatives_per_component': 10, 'multilevel_base_size': 512, 'multilevel_ep

Construct MST graph:   0%|          | 0/32 [00:00<?, ?steps/s]

Building RP forest with 8 trees...
Initializing graph from 133030 leaves...
Running NN-descent for up to 21 iterations...
Iteration 1: 160291028 updates
Iteration 2: 66469572 updates
Iteration 3: 13732565 updates
Iteration 4: 2938362 updates
Iteration 5: 669853 updates
Iteration 6: 165644 updates
Iteration 7: 45895 updates
Converged after 7 iterations

=== Timing Breakdown ===
Forest building:    5989.516ms
Leaf initialization:18962.255ms
Candidate building: 10839.004ms (7 iters)
Update iterations:  161627.538ms (7 iters)
Total measured:     197418.313ms
Tue Nov 24 11:46:33 2026 MST graph construction complete: 1206.150s


Multilevel coarsening:   0%|          | 0/1988195 [00:00<?, ?nodes/s]

## 5. Compare runtimes and display the 2D maps

In [ ]:
runtime = pd.DataFrame([
    {"method": "IMSTE tuning", "seconds": tuning_seconds},
    {"method": "UMAP", "seconds": umap_seconds},
    {"method": "IMSTE", "seconds": imste_seconds},
])
runtime["runtime"] = runtime["seconds"].map(lambda x: f"{x / 60:.1f} min" if x >= 60 else f"{x:.1f} s")
print(f"Input data: {X.shape[0]:,} instances × {X.shape[1]:,} original dimensions")
display(runtime)

Input data: 1,988,195 instances × 384 original dimensions


,method,seconds,runtime
0,IMSTE tuning,0.000000,0.0 s
1,UMAP,149.432625,2.5 min
2,IMSTE,290.711213,4.8 min


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_hex
from ipywidgets import Dropdown, VBox, interactive_output
from imste.visualization import render_datashader_embedding

COLOR_OPTIONS = [
    "source_batch", "language", "hour", "weekday", "retweet", "retweet_count", "hashtag", "engagement",
]

LANGUAGE_NAMES = {
    "ar": "Arabic", "bg": "Bulgarian", "bn": "Bengali", "ca": "Catalan",
    "cs": "Czech", "cy": "Welsh", "da": "Danish", "de": "German",
    "el": "Greek", "en": "English", "es": "Spanish", "et": "Estonian",
    "eu": "Basque", "fa": "Persian", "fi": "Finnish", "fr": "French",
    "ga": "Irish", "gl": "Galician", "gu": "Gujarati", "he": "Hebrew",
    "hi": "Hindi", "hr": "Croatian", "ht": "Haitian Creole", "hu": "Hungarian",
    "id": "Indonesian", "in": "Indonesian", "is": "Icelandic", "it": "Italian",
    "iw": "Hebrew", "ja": "Japanese", "kn": "Kannada", "ko": "Korean",
    "lt": "Lithuanian", "lv": "Latvian", "ml": "Malayalam", "mr": "Marathi",
    "ms": "Malay", "mt": "Maltese", "ne": "Nepali", "nl": "Dutch",
    "no": "Norwegian", "pa": "Punjabi", "pl": "Polish", "pt": "Portuguese",
    "ro": "Romanian", "ru": "Russian", "si": "Sinhala", "sk": "Slovak",
    "sl": "Slovenian", "sr": "Serbian", "sv": "Swedish", "ta": "Tamil",
    "te": "Telugu", "th": "Thai", "tl": "Tagalog", "tr": "Turkish",
    "uk": "Ukrainian", "ur": "Urdu", "vi": "Vietnamese", "zh": "Chinese",
    "zh-cn": "Simplified Chinese", "zh-tw": "Traditional Chinese",
    "und": "Undetermined", "unknown": "Unknown",
}


def legend_name(name, color_by):
    return LANGUAGE_NAMES.get(name, name) if color_by == "language" else name


def labels_for_color_by(color_by):
    timestamps = pd.to_datetime(tweets["created_at"], format="mixed", errors="coerce", utc=True)
    if color_by == "source_batch":
        color_values = tweets["source_batch"].fillna("unknown").astype(str)
    elif color_by == "language":
        color_values = tweets["lang"].fillna("unknown").astype(str)
    elif color_by == "hour":
        color_values = timestamps.dt.hour.map(
            lambda x: f"{int(x):02d}:00" if pd.notna(x) else "unknown"
        )
    elif color_by == "weekday":
        color_values = timestamps.dt.day_name().fillna("unknown")
    elif color_by == "retweet":
        color_values = tweets["is_retweet"].map(
            {True: "retweet", False: "original"}
        ).fillna("unknown")
    elif color_by == "retweet_count":
        counts = pd.to_numeric(tweets["retweet_count"], errors="coerce").fillna(0).clip(lower=0)
        color_values = pd.cut(
            counts,
            bins=[-0.5, 0.5, 1.5, 5.5, 20.5, 100.5, np.inf],
            labels=["0", "1", "2–5", "6–20", "21–100", ">100"],
        ).astype(str)
    elif color_by == "hashtag":
        hashtags = tweets["top_hashtag"].fillna("(no hashtag)").astype(str)
        no_hashtag = hashtags.eq("(no hashtag)")
        hashtag_counts = hashtags[~no_hashtag].value_counts()
        top_hashtags = set(hashtag_counts.head(TOP_HASHTAGS).index)
        color_values = hashtags.where(
            no_hashtag | hashtags.isin(top_hashtags), "Other hashtags"
        ).replace("(no hashtag)", "No hashtag")
    elif color_by == "engagement":
        retweets = pd.to_numeric(tweets["retweet_count"], errors="coerce")
        favorites = pd.to_numeric(tweets["favorite_count"], errors="coerce")
        engagement = (retweets + favorites).replace([np.inf, -np.inf], np.nan).fillna(0)
        if engagement.nunique() < 2:
            value = engagement.iloc[0] if len(engagement) else 0
            color_values = pd.Series(
                f"all tweets: {value:g} engagement", index=tweets.index, dtype="string"
            )
        else:
            color_values = pd.qcut(
                np.log1p(engagement), q=5, duplicates="drop"
            ).astype(str)
    else:
        raise ValueError(f"Unknown color field: {color_by}")

    color_values = color_values.astype(str)
    counts = color_values.value_counts()
    if color_by in {"source_batch", "language", "hashtag"} and len(counts) > 20:
        keep = set(counts.head(19).index)
        color_values = color_values.where(color_values.isin(keep), "other")
    return pd.Categorical(color_values)


def plot_embeddings(color_by):
    selected_labels = labels_for_color_by(color_by)
    category_order = list(selected_labels.categories)
    if color_by == "hour":
        # Cyclic day scale: black at midnight, yellow at midday, black again.
        hour_cmap = LinearSegmentedColormap.from_list(
            "black_red_yellow_black",
            [(0.0, "#000000"), (0.25, "#d7191c"), (0.5, "#ffff00"),
             (0.75, "#d7191c"), (1.0, "#000000")],
        )
        color_key = {}
        for name in category_order:
            if name == "unknown":
                color_key[name] = "#808080"
            else:
                hour = int(name[:2])
                color_key[name] = to_hex(hour_cmap(hour / 23))
    elif len(category_order) == 2:
        # Keep binary comparisons high-contrast; categorical order puts
        # "original" before "retweet", so retweets appear in red.
        binary_colors = ["#1f77b4", "#d62728"]  # blue, red
        color_key = dict(zip(category_order, binary_colors))
    else:
        # tab10 uses distinct hues for small category sets; tab20 adds capacity
        # when more than ten categories need colors.
        palette = plt.get_cmap("tab10" if len(category_order) <= 10 else "tab20")
        color_key = {
            name: to_hex(palette(i % palette.N))
            for i, name in enumerate(category_order)
        }
    label_values = selected_labels.astype(str)
    fig, axes = plt.subplots(2, 1, figsize=(12, 14), constrained_layout=True)
    for ax, title, coordinates in zip(axes, ("UMAP", "IMSTE"), (Z_umap, Z_imste)):
        image = render_datashader_embedding(
            coordinates[:, :2], label_values, width=1200, height=900,
            color_key=color_key, min_alpha=20, background="white", spread_px=1,
        )
        ax.imshow(image.to_pil())
        ax.set_title(f"{title} · colored by {color_by}")
        ax.set_axis_off()
    handles = [
        plt.Line2D(
            [0], [0], marker="o", linestyle="", color=color_key[name],
            label=legend_name(name, color_by),
        )
        for name in category_order
    ]
    fig.legend(handles=handles, loc="outside lower center",
               ncol=min(6, len(handles)), frameon=False)
    display(fig)
    plt.close(fig)

color_by_selector = Dropdown(
    options=COLOR_OPTIONS,
    value=COLOR_BY if COLOR_BY in COLOR_OPTIONS else "source_batch",
    description="Color by:",
    layout={"width": "250px"},
)
plot_output = interactive_output(plot_embeddings, {"color_by": color_by_selector})
display(VBox([color_by_selector, plot_output]))
